# Modélisation Deep Learning - Amazon Sentiment

Ce notebook implémente un modèle de classification de sentiment utilisant TensorFlow/Keras.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalMaxPooling1D
import os

## Chargement et Préparation des données

In [2]:
df = pd.read_csv(os.path.join('..', 'data', 'amazon_reviews.csv'))

# Encodage des labels
le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])
num_classes = len(le.classes_)

X = df['review_body'].values
y = df['sentiment_encoded'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Tokenisation et Padding

In [3]:
max_words = 5000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)

## Construction du Modèle

In [4]:
model = Sequential([
    Embedding(max_words, 64, input_length=max_len),
    LSTM(64, return_sequences=True),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\Franck\.gemini\antigravity\scratch\Projets\amazon-sentiment\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Entraînement

In [5]:
history = model.fit(X_train_seq, y_train, epochs=5, batch_size=32, 
                    validation_data=(X_test_seq, y_test), verbose=1)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 40ms/step - accuracy: 0.8940 - loss: 0.2744 - val_accuracy: 1.0000 - val_loss: 6.0760e-04
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 1.0000 - loss: 0.0030 - val_accuracy: 1.0000 - val_loss: 5.2831e-05
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.9998 - loss: 0.0018 - val_accuracy: 1.0000 - val_loss: 1.2989e-05
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 1.0000 - loss: 6.4879e-04 - val_accuracy: 1.0000 - val_loss: 2.7822e-06
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.9998 - loss: 7.8825e-04 - val_accuracy: 1.0000 - val_loss: 1.7092e-06


## Visualisation des Performances avec Plotly

In [6]:
history_df = pd.DataFrame(history.history)

fig_acc = go.Figure()
fig_acc.add_trace(go.Scatter(y=history_df['accuracy'], name='Train Accuracy'))
fig_acc.add_trace(go.Scatter(y=history_df['val_accuracy'], name='Val Accuracy'))
fig_acc.update_layout(title='Précision du Modèle', xaxis_title='Epoque', yaxis_title='Précision')
fig_acc.show()

fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(y=history_df['loss'], name='Train Loss'))
fig_loss.add_trace(go.Scatter(y=history_df['val_loss'], name='Val Loss'))
fig_loss.update_layout(title='Perte du Modèle', xaxis_title='Epoque', yaxis_title='Perte')
fig_loss.show()